In [ ]:
## Load packages; set plotting parameters
import os
import scanpy as sc
import anndata as ad
import pandas as pd
import numpy as np
from collections import Counter
from matplotlib import cm,colors
import seaborn as sns
from harmonypy import run_harmony   


In [ ]:
## Load in 10X matrix per sample and merge into a single adata table
sample_meta_path = "../input_metadata//sample_data.txt"

meta = pd.read_csv(sample_meta_path, sep="\t")
required_cols = {"sample_id", "RNA_matrix"}
missing = required_cols - set(meta.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

adata_list = []

for _, row in meta.iterrows():
    sample_id = row["sample_id"]
    data_dir = row["RNA_matrix"]

    adata = sc.read_10x_mtx(
        data_dir,
        var_names="gene_symbols",
        cache=False
    )

    adata.var_names_make_unique()
    adata.obs_names = [f"{sample_id}_{cell}" for cell in adata.obs_names]
    adata.obs["sample"] = sample_id

    adata_list.append(adata)

sample_categories = [adata.obs["sample"].unique()[0] for adata in adata_list]

combined = adata_list[0].concatenate(
    adata_list[1:], 
    batch_key="sample",
    batch_categories=sample_categories,
    index_unique=None
)

In [ ]:
## Running QC, filterings and normalization
combined.var["mt"] = combined.var_names.str.startswith("MT-")
sc.pp.calculate_qc_metrics(combined, qc_vars=["mt"], inplace=True)

qc_before = (
    combined.obs
    .groupby("sample")
    .agg(
        n_cells_before=("sample", "size"),
        median_genes_before=("n_genes_by_counts", "median"),
        median_mt_before=("pct_counts_mt", "median")
    )
    .reset_index()
)

combined = combined[
    (combined.obs.n_genes_by_counts > 200) &
    (combined.obs.n_genes_by_counts < 5000) &
    (combined.obs.pct_counts_mt < 20),
    :
]

qc_after = (
    combined.obs
    .groupby("sample")
    .agg(
        n_cells_after=("sample", "size"),
        median_genes_after=("n_genes_by_counts", "median"),
        median_mt_after=("pct_counts_mt", "median")
    )
    .reset_index()
)

qc_summary = (
    qc_before
    .merge(qc_after, on="sample", how="left")
)

# add percent retained
qc_summary["pct_cells_retained"] = (
    qc_summary["n_cells_after"] / qc_summary["n_cells_before"] * 100
)

# Preprocessing
sc.pp.normalize_total(combined, target_sum=1e4)
sc.pp.log1p(combined)
sc.pp.highly_variable_genes(combined, n_top_genes=4000, subset=False)
hvg_genes = combined.var[combined.var['highly_variable']].index
sc.pp.scale(combined, max_value=10)
sc.tl.pca(combined, svd_solver='arpack', #use_highly_variable=True)
          mask_var="highly_variable")

In [ ]:
## Run Harmony and run umap
ho = run_harmony(combined.obsm['X_pca'],meta_data = combined.obs, vars_use = ['sample'],
                            random_state = 0, nclust = 50, max_iter_harmony = 20)
combined.obsm['X_pca_harmony'] = ho.Z_corr.T
sc.pp.neighbors(combined, use_rep = 'X_pca_harmony',random_state=0)#, n_neighbors=20, n_pcs=30)
sc.tl.leiden(combined, resolution=0.6, key_added="leiden_harmony",random_state=0)
#sc.tl.leiden(combined, resolution=0.5, key_added="leiden_harmony2",random_state=0)
sc.tl.umap(combined,random_state=0)
sc.tl.tsne(combined,use_rep="X_pca_harmony",random_state=0,perplexity=30,n_jobs=32)

sc.pl.umap(combined, color=["sample", "leiden_harmony"],legend_fontsize=8,save = "_44sample_rna_k50_umap.pdf")
sc.pl.tsne(combined,color=["sample", "leiden_harmony"],legend_fontsize=8,save="_44sample_rna_k50_tsne.pdf")

combined.obs["UMAP1"] = combined.obsm["X_umap"][:, 0]
combined.obs["UMAP2"] = combined.obsm["X_umap"][:, 1]

combined.obs["TSNE1"] = combined.obsm["X_tsne"][:, 0]
combined.obs["TSNE2"] = combined.obsm["X_tsne"][:, 1]

In [ ]:
## Run consensus clustering
n_runs = 20
resolution = 0.6

for seed in range(n_runs):
    sc.pp.neighbors(combined, use_rep="X_pca_harmony", random_state=seed)
    sc.tl.leiden(
        combined,
        resolution=resolution,
        key_added=f"leiden_{seed}"
    )
    print("run " + str(seed) + " finished!")

clustering = combined.obs
n_runs=20
ref = clustering["leiden_harmony"]

for i in range(0, n_runs):
    cur = clustering[f"leiden_{i}"]
    mapping = {}
    for c in cur.unique():
        overlap = ref[cur == c]
        mapping[c] = overlap.value_counts().idxmax()
    clustering[f"leiden2_{i}"] = cur.map(mapping)

leiden_cols = [f"leiden2_{i}" for i in range(n_runs)]
labels_df = clustering[leiden_cols]

def consensus_label(row):
    return row.value_counts().idxmax()

clustering["consensus_cluster"] = labels_df.apply(consensus_label, axis=1)

old = clustering["consensus_cluster"]

old_levels = np.sort(old.unique())

mapping = {old_label: new_label 
           for new_label, old_label in enumerate(old_levels)}

clustering["consensus_cluster"] = old.map(mapping)
clustering = clustering.set_index("Unnamed: 0")
combined.obs["rna_consensus_cluster"] = labels_df.apply(consensus_label, axis=1)

clustering.to_csv(
    "tmp_dataset/rna_consensus_clustering.tsv",
    sep="\t",
    index=True   # keep cell barcodes as first column
)
combined.write("tmp_dataset/combined_data_rna.h5ad")